In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
df = spark.read.table("medillian_upsert.silver.products")
df.display()

In [0]:
df = df.dropDuplicates(subset = ['product_id'])
df.count()

In [0]:
# df =  df.withColumn("DimProductsKey",monotonically_increasing_id() + lit(1))
# df.show()
w = Window.orderBy("product_id")
df = df.withColumn("DimProductsKey", row_number().over(w))
df.display()

In [0]:
# Check if the DimCustomers table exists
table_exists = spark.catalog.tableExists("medillian_upsert.gold.DimProducts")

if table_exists:
    df_old = spark.sql('''select DimProductsKey, product_id, create_date, update_date
                       from medillian_upsert.gold.DimProducts ''')
else:
    # Return empty DataFrame with correct schema for initial load or when table doesn't exist
    df_old = spark.sql('''select cast(0 as bigint) as DimProductsKey, 
                                 cast(0 as bigint) as product_id, 
                                 cast(null as timestamp) as create_date, 
                                 cast(null as timestamp) as update_date
                       from medillian_upsert.silver.products where 1 = 0 ''')

In [0]:
df_old.show()

In [0]:
df_old = df_old.withColumnRenamed("DimProductsKey", "old_DimProductsKey")\
                .withColumnRenamed("product_id", "old_product_id")\
                .withColumnRenamed("create_date", "old_create_date")\
                .withColumnRenamed("update_date", "old_update_date")

In [0]:
df_old.show()

In [0]:
df_join = df.join(df_old, df['product_id'] == df_old['old_product_id'], 'left')
df_join.display()

In [0]:
df_new = df_join.filter(df_join['old_DimProductsKey'].isNull())

In [0]:
df_old = df_join.filter(df_join['old_DimProductsKey'].isNotNull())

In [0]:
df_old.show()

In [0]:
# Dropping all the columns which are not required
df_old = df_old.drop('old_DimProductsKey', 'old_product_id', 'old_update_date')

# Renaming "old_create_date"  column to "create_date"
df_old = df_old.withColumnRenamed('old_create_date', 'create_date')
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

#Recreating "update_date" column with current timestamp
df_old = df_old.withColumn('update_date', current_timestamp())


In [0]:
df_old.show()

In [0]:
df_new.display()

In [0]:
# Dropping all the columns which are not required
df_new = df_new.drop('old_DimProductsKey', 'old_product_id', 'old_update_date','old_create_date')

#Recreating "update_date",current_date column with current timestamp
df_new = df_new.withColumn('update_date', current_timestamp())
df_new = df_new.withColumn('create_date', current_timestamp())


In [0]:
df_new.display()

In [0]:
# df_new =  df_new.withColumn("DimProductsKey",monotonically_increasing_id() + lit(1))
# df_new.display()

w = Window.orderBy("product_id")
df = df.withColumn("DimProductsKey", row_number().over(w))
df.display()


In [0]:
if spark.catalog.tableExists("medillian_upsert.gold.DimProducts"):
    max_surrogate_key = (
        spark.sql("""
            SELECT MAX(DimProductsKey) AS max_surrogate_key
            FROM medillian_upsert.gold.DimProducts
        """)
        .collect()[0]["max_surrogate_key"]
    )

    if max_surrogate_key is None:
        max_surrogate_key = 0
else:
    max_surrogate_key = 0

In [0]:
print(max_surrogate_key)

In [0]:
# df_new = df_new.withColumn("DimProductsKey", lit(max_surrogate_key) + col("DimProductsKey"))

w = Window.orderBy("product_id")
df_new = df_new.withColumn("DimProductsKey", lit(max_surrogate_key) + row_number().over(w))
df_new.display()

In [0]:
df_new.display()

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.show()

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("medillian_upsert.gold.DimProducts"):
    dlt_obj = DeltaTable.forName(spark, "medillian_upsert.gold.DimProducts")

    dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.product_id = src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .saveAsTable("medillian_upsert.gold.DimProducts")

In [0]:
%sql
select * from medillian_upsert.gold.DimProducts